# Structured streaming Demo

In [1]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

from pyspark.sql.functions import window, col, count

# Convert timestamp (Unix epoch in seconds) to timestamp type
from pyspark.sql.functions import from_unixtime




In [2]:
# initialize the spark session
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0") \
    .getOrCreate()


In [3]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


In [4]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

In [5]:
# A simple aggregation of raw data by number of bot edits
windowed_counts_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

In [6]:

# Use console output (goes to Docker logs)
streaming_query_df = windowed_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

## Enrichment of the demo

Each new query below is also written to the console, so its output appears in `docker logs -f pyspark_notebook` next to the demo output.

New columns used by the aggregations:
- `event_time`: event time of the edit (same conversion as the demo)
- `edit_size`: size of the edit in bytes, `length.new - length.old` (negative when content is removed)

In [7]:
from pyspark.sql import functions as F

# Edits with the event time and the size of each edit
edits_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .withColumn("edit_size", col("length.new") - col("length.old"))

## Exploration 1: tumbling windows

Window of 10 minutes, no slide: the windows follow each other without overlapping, so every edit is counted in exactly one window.

For each window and each wiki: number of edits, total, average and maximum edit size.

In [8]:
tumbling_df = edits_df \
    .groupBy(
        window(col("event_time"), "10 minutes"),
        col("wiki")
    ) \
    .agg(
        count("*").alias("edit_count"),
        F.sum("edit_size").alias("total_edit_size"),
        F.round(F.avg("edit_size"), 1).alias("avg_edit_size"),
        F.max("edit_size").alias("max_edit_size")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("wiki"),
        col("edit_count"),
        col("total_edit_size"),
        col("avg_edit_size"),
        col("max_edit_size")
    ) \
    .orderBy(col("window_start").desc(), col("wiki"))

tumbling_query = tumbling_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

## Exploration 2: overlapping (sliding) windows

Window of 30 minutes sliding every 10 minutes: a new window starts every 10 minutes and the windows overlap, so every edit is counted in 3 windows.

For each window, bots vs humans: number of edits and average edit size.

In [9]:
sliding_df = edits_df \
    .groupBy(
        window(col("event_time"), "30 minutes", "10 minutes"),
        col("bot")
    ) \
    .agg(
        count("*").alias("edit_count"),
        F.round(F.avg("edit_size"), 1).alias("avg_edit_size")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count"),
        col("avg_edit_size")
    ) \
    .orderBy(col("window_start").desc(), col("bot"))

sliding_query = sliding_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()